# Friend C — B31 duplicate-node clustering (Mac CPU)

This notebook uses Friend A's compact `raw/dc/ckpt + GT` package. It does not load images, model weights, CUDA, or PyTorch. The run is resumable per video.

To stop cleanly while the long cell is running, create the `STOP` file printed by the configuration cell. The current video finishes; delete `STOP` and rerun the long cell to continue.

In [ ]:
from pathlib import Path
import os, sys

def find_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'PROMPT_FRIEND_C.md').is_file() and (candidate / 'tools' / 'public_local.py').is_file():
            return candidate
    raise FileNotFoundError('Repository root not found; start Jupyter inside the cloned repository')

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT))
from experiments.local_friend_c_b31_cluster.runner import Config, compact_status, preflight, run_pipeline
print('Repository:', ROOT)

In [ ]:
cache_root = Path(os.environ.get('BIOHUB_LOCAL_CACHE', ROOT / 'friend_c_b31_data' / 'cache')).expanduser()
data_dir = Path(os.environ.get('BIOHUB_DATA_DIR', ROOT / 'friend_c_b31_data' / 'train')).expanduser()
output_dir = Path(os.environ.get('BIOHUB_B31_OUTPUT', ROOT / 'outputs' / 'local_friend_c_b31_cluster')).expanduser()

cfg = Config(
    cache_root=cache_root,
    data_dir=data_dir,
    output_dir=output_dir,
    tag=os.environ.get('BIOHUB_B31_TAG', 'friend_a_b20_exp29'),
    radii_um=(0.0, 0.5, 1.0, 1.5, 2.0),
    stop_file=output_dir / 'STOP',
    expected_videos=199,
)
print('Cache:', cfg.cache_dir)
print('GT:', cfg.data_dir)
print('Outputs:', cfg.output_dir)
print('Clean stop file:', cfg.stop_file)

In [ ]:
preflight(cfg)

## Full resumable run

This cell performs the distance audit first. If fewer than 1% of raw nodes have a neighbour below 2 µm, it records the registered no-lever verdict and stops. Otherwise it validates the radius-zero control and evaluates all four clustering radii. Rerunning it reuses completed work.

In [ ]:
result = run_pipeline(cfg)
result

In [ ]:
compact_status(cfg)